# DipDECK Colab deduplicated sensitivity runner

This is `dipdeck_colab.ipynb` with one change: the scientific grid and reference run with `--dedup`, on one record per exact-duplicate group (1,470 GraphPart-retained records, `data.deduplicate`'s survivor rule), and write to `scientific_grid_dedup` and `scientific_reference_dedup`. Settings, folds, seeds, epochs, source pins and preflight checks are unchanged.

This notebook uses Google Drive for the project and its outputs. Upload this project folder to Google Drive as `MyDrive/explaining-plm-latent-spaces` (or set `PROJECT_DIR`), select a GPU runtime, and run all cells.

The notebook copies the project from Drive and checks its data archive. It tests the selected ClustPy revision on CPU, then runs a CUDA contract test. `RUN_SCIENTIFIC_GRID` runs the 20-setting, 10-seed, five-fold grid. `RUN_SCIENTIFIC_REFERENCE` runs the predeclared paper-default full-data reference across the same ten seeds. Each route validates and archives only its own generated tree.


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/explaining-plm-latent-spaces"
CLUSTPY_COMMIT = "82548dc35a01d221474fbb44908b3621d4b97616"
EXPECTED_UPSTREAM_DIPDECK_SHA256 = "c567d24c1722087a75c3eb56f17fe91152b6f818148083d908086b9fc78d438a"
EXPECTED_DIPDECK_SHA256 = "cfbf629ab341bdbb5967af232c24cfc3f2e5fb09cd082fb28b317c45738b7420"
EXPECTED_DATA_ZIP_SHA256 = "3ed7857f71e4c63939ffaee885b485bdb01555d8eced9abc18b567c3711a08e2"
DRIVE_ROOT = PROJECT_DIR
DATA_ZIP = f"{PROJECT_DIR}/data/phosphatase/dipdeck_phosphatase_data.zip"
RUN_PHOSPHATASE_SMOKE = False
RUN_SCIENTIFIC_GRID = True
RUN_SCIENTIFIC_REFERENCE = True
SMOKE_SEED = 42
SMOKE_PRETRAIN_EPOCHS = 3
SMOKE_CLUSTERING_EPOCHS = 3

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import zipfile

repo = Path("/content/project").resolve()
clustpy_source = Path("/content/ClustPy").resolve()
drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)

for target in (repo, clustpy_source):
    if target.parent != Path("/content"):
        raise RuntimeError(f"Refusing to replace unexpected path: {target}")
    if target.exists():
        shutil.rmtree(target)

project_dir = Path(PROJECT_DIR)
if not (project_dir / "src/algo/dipdeck/dipdeck_standalone.py").is_file():
    raise FileNotFoundError(f"Upload this project folder to Google Drive at {project_dir}, then rerun.")
for part in ("src", "data", "results/graphpart", "results/ec_audit"):
    shutil.copytree(project_dir / part, repo / part)
subprocess.run(["git", "clone", "https://github.com/collinleiber/ClustPy.git", str(clustpy_source)], check=True)
subprocess.run(["git", "-C", str(clustpy_source), "checkout", CLUSTPY_COMMIT], check=True)

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
source_text = dipdeck_source.read_text(encoding="utf-8")
upstream_sha256 = hashlib.sha256(source_text.encode()).hexdigest()
if upstream_sha256 != EXPECTED_UPSTREAM_DIPDECK_SHA256:
    raise RuntimeError(
        f"Unexpected upstream DipDECK source: expected {EXPECTED_UPSTREAM_DIPDECK_SHA256}, "
        f"got {upstream_sha256}"
    )
for old, new in (
    ("self.labels.to(device)", "self.labels = self.labels.to(device)"),
    ("self.cluster_centers.to(device)", "self.cluster_centers = self.cluster_centers.to(device)"),
    ("self.dip_matrix.to(device)", "self.dip_matrix = self.dip_matrix.to(device)"),
):
    if source_text.count(old) != 1:
        raise RuntimeError(f"CUDA patch target is absent or ambiguous: {old}")
    source_text = source_text.replace(old, new)
dipdeck_source.write_text(source_text, encoding="utf-8")
patched_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if patched_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"Patched DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {patched_sha256}"
    )

repo_commit = "project-folder"

data_zip = Path(DATA_ZIP)
if not data_zip.is_file():
    raise FileNotFoundError(
        f"Upload dipdeck_phosphatase_data.zip to {data_zip.parent} in Google Drive, then rerun."
    )
data_zip_sha256 = hashlib.sha256(data_zip.read_bytes()).hexdigest()
if data_zip_sha256 != EXPECTED_DATA_ZIP_SHA256:
    raise RuntimeError(
        f"Data ZIP mismatch: expected {EXPECTED_DATA_ZIP_SHA256}, got {data_zip_sha256}"
    )
with zipfile.ZipFile(data_zip) as archive:
    archive.extractall(repo)

required = [
    repo / "data/phosphatase/phosphatase/tmp/prot_t5.h5",
    repo / "data/phosphatase/phosphatase/data_standard.parquetbundle",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Data ZIP has the wrong layout or is incomplete: {missing}")

os.chdir(repo)
print(f"Repository: {repo}")
print(f"Project copied from: {PROJECT_DIR}")
print(f"Data ZIP: {data_zip}")
print(f"Data ZIP SHA256: {data_zip_sha256}")
print(f"ClustPy revision: {CLUSTPY_COMMIT}")

In [ ]:
COLAB_PINS = {
    "Cython": "3.3.0",
    "numpy": "2.1.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "h5py": "3.16.0",
    "pyarrow": "25.0.1",
    "tqdm": "4.70.0",
    "pytest": "9.1.1",
    "psutil": "7.2.2",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
}

import os
import subprocess
import sys

os.chdir("/content")
install_names = [
    "Cython", "numpy", "scipy", "scikit-learn", "pandas",
    "h5py", "pyarrow", "tqdm", "pytest", "psutil",
]
requirements = [f"{name}=={COLAB_PINS[name]}" for name in install_names]
install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", *requirements],
    check=True, capture_output=True, text=True,
)
print(install_result.stdout)
if install_result.stderr:
    print(install_result.stderr, file=sys.stderr)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "--force-reinstall", "--no-cache-dir", "--editable", str(clustpy_source),
] , check=True)

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True,
)
print(pip_check.stdout)
if pip_check.stderr:
    print(pip_check.stderr, file=sys.stderr)
if pip_check.returncode != 0:
    print("pip check reported the Colab base-environment warning above.", file=sys.stderr)

In [ ]:
import hashlib
import importlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

for module_name in (
    "Cython", "h5py", "pandas", "psutil", "pyarrow", "pytest", "scipy", "sklearn", "tqdm",
):
    importlib.import_module(module_name)

import numpy as np
import torch
import torchvision
from threadpoolctl import threadpool_info

if not (clustpy_source / "clustpy/__init__.py").is_file():
    raise FileNotFoundError(f"ClustPy package is missing from the verified checkout: {clustpy_source}")
sys.path.insert(0, str(clustpy_source))
import clustpy
from clustpy.deep import DipDECK
if Path(clustpy.__file__).resolve().parent != (clustpy_source / "clustpy").resolve():
    raise RuntimeError(f"Imported ClustPy is not the verified checkout: {clustpy.__file__}")

observed_versions = {name: metadata.version(name) for name in COLAB_PINS}
version_mismatches = {
    name: {"expected": expected, "observed": observed_versions[name]}
    for name, expected in COLAB_PINS.items()
    if observed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f"Colab dependency mismatch: {version_mismatches}")

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
dipdeck_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if dipdeck_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {dipdeck_sha256}"
    )

environment = {
    "python": platform.python_version(),
    "project_commit": repo_commit,
    "clustpy_version": metadata.version("clustpy"),
    "clustpy_path": clustpy.__file__,
    "clustpy_commit": CLUSTPY_COMMIT,
    "dipdeck_sha256": dipdeck_sha256,
    "package_versions": observed_versions,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_num_threads": torch.get_num_threads(),
    "torch_num_interop_threads": torch.get_num_interop_threads(),
    "thread_environment": {
        name: os.environ.get(name)
        for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    },
    "threadpool_info": threadpool_info(),
}
print(json.dumps(environment, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU, then run all again.")

In [ ]:
cpu_env = os.environ.copy()
cpu_env["CLUSTPY_DEVICE"] = "cpu"
cpu_env["RUN_DIPDECK_SELECTED_RELEASE_TESTS"] = "1"
cpu_env["DIPDECK_CLUSTPY_SOURCE_ROOT"] = str(clustpy_source)
cpu_env["PYTHONPATH"] = str(clustpy_source) + os.pathsep + cpu_env.get("PYTHONPATH", "")
subprocess.run([
    sys.executable, "-c",
    "from clustpy.deep.tests import _helpers_for_tests; "
    "print('Upstream test helpers:', _helpers_for_tests.__file__)",
], cwd=repo, env=cpu_env, check=True)
test_result = subprocess.run([
    sys.executable, "-m", "pytest",
    "/content/ClustPy/clustpy/deep/tests/test_dipdeck.py",
    "/content/ClustPy/clustpy/utils/tests/test_diptest.py",
    "-q",
], cwd=repo, env=cpu_env, capture_output=True, text=True)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr, file=sys.stderr)
test_result.check_returncode()

In [ ]:
from sklearn.datasets import make_blobs

X_test, _ = make_blobs(n_samples=240, n_features=16, centers=4, random_state=SMOKE_SEED)
X_test = X_test.astype(np.float32)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from src.algo.dipdeck import dipdeck_standalone as standalone_module
cuda_config = standalone_module.SmokeConfig(
    n_clusters_init=8,
    pretrain_epochs=1,
    clustering_epochs=1,
    batch_size=64,
    seed=SMOKE_SEED,
)
cuda_test, _ = standalone_module._create_dipdeck(cuda_config, "cuda", clustpy_source)
with standalone_module._count_dipdeck_clustering_epochs(cuda_test) as epoch_updates:
    cuda_test.fit(X_test)
if len(cuda_test.labels_) != len(X_test) or not np.isfinite(cuda_test.cluster_centers_).all():
    raise RuntimeError("CUDA contract test returned invalid outputs")
standalone_module._validate_fitted_outputs(
    cuda_test, X_test, np.asarray([f"probe_{i:03d}" for i in range(len(X_test))]),
    cuda_config.embedding_size,
)
if epoch_updates is None:
    raise RuntimeError("CUDA contract did not record clustering epochs")
standalone_module._validate_realised_clustering_epochs(
    epoch_updates[0], "measured_selected_source_tqdm", cuda_config.clustering_epochs,
    cuda_test.n_clusters_,
)
print(f"CUDA contract passed with {cuda_test.n_clusters_} clusters and "
      f"{epoch_updates[0]} realised clustering epochs")

In [ ]:
from datetime import datetime, timezone
import time

if RUN_PHOSPHATASE_SMOKE:
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from src.data import load_dataset

    dataset = load_dataset("phosphatase")
    X = np.asarray(dataset.X, dtype=np.float32)
    embedding_ids = np.asarray(dataset.ids, dtype=str)
    annotation_ids = dataset.ann["protein_id"].astype(str).to_numpy()
    if X.shape != (1587, 1024) or not np.isfinite(X).all():
        raise RuntimeError(f"Unexpected phosphatase matrix: {X.shape}")
    if len(np.unique(embedding_ids)) != len(embedding_ids):
        raise RuntimeError("Phosphatase embedding IDs are not unique")
    if not np.array_equal(embedding_ids, annotation_ids):
        raise RuntimeError("Phosphatase annotation rows are not aligned to embedding IDs")
    if dataset.subset_hash != "1906375a02bc":
        raise RuntimeError(f"Unexpected phosphatase subset hash: {dataset.subset_hash}")
    print(
        f"Data contract passed: shape={X.shape}, unique_ids={len(embedding_ids)}, "
        f"subset_hash={dataset.subset_hash}"
    )

    output_dir = drive_root / "results/dipdeck_colab_smoke"
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = DipDECK(
        n_clusters_init=35,
        dip_merge_threshold=0.9,
        embedding_size=5,
        batch_size=256,
        pretrain_epochs=SMOKE_PRETRAIN_EPOCHS,
        clustering_epochs=SMOKE_CLUSTERING_EPOCHS,
        max_cluster_size_diff_factor=2,
        pval_strategy="table",
        device=torch.device("cuda"),
        random_state=SMOKE_SEED,
    )
    model.fit(X)
    elapsed = time.perf_counter() - started
    labels = np.asarray(model.labels_)
    if labels.shape != (len(dataset),) or not np.isfinite(model.cluster_centers_).all():
        raise RuntimeError("Phosphatase smoke test returned invalid outputs")

    result = {
        **environment,
        "run_kind": "reduced-epoch feasibility test, not a scientific result",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "phosphatase",
        "n": len(dataset),
        "n_dims": X.shape[1],
        "subset_hash": dataset.subset_hash,
        "seed": SMOKE_SEED,
        "pretrain_epochs": SMOKE_PRETRAIN_EPOCHS,
        "clustering_epochs": SMOKE_CLUSTERING_EPOCHS,
        "n_clusters": int(model.n_clusters_),
        "elapsed_seconds": elapsed,
        "peak_cuda_memory_bytes": int(torch.cuda.max_memory_allocated()),
    }
    (output_dir / "environment_and_result.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8",
    )
    np.savez_compressed(
        output_dir / "assignment.npz",
        protein_id=embedding_ids,
        label=labels,
    )
    print(json.dumps(result, indent=2))
    print(f"Saved smoke outputs to {output_dir}")
else:
    print("Phosphatase smoke test skipped by configuration.")

In [ ]:
standalone = repo / "src/algo/dipdeck/dipdeck_standalone.py"
if (RUN_SCIENTIFIC_GRID or RUN_SCIENTIFIC_REFERENCE) and not standalone.is_file():
    raise FileNotFoundError(f"Standalone is missing: {standalone}")
import sys
output_root = (drive_root / "results/dipdeck").resolve()
standalone_env = os.environ.copy()
standalone_env["DIPDECK_OUTPUT_ROOT"] = str(output_root)

def run_scientific(command, validator, owned_directory, archive_name):
    if owned_directory.parent != output_root:
        raise RuntimeError(f"Refusing to replace unexpected output: {owned_directory}")
    if owned_directory.exists():
        shutil.rmtree(owned_directory)
    subprocess.run(
        [sys.executable, str(standalone), command, "--dedup", "--device", "cuda",
         "--clustpy-source-root", str(clustpy_source)],
        cwd=repo, env=standalone_env, check=True,
    )
    subprocess.run(
            [sys.executable, str(standalone), validator, "--dedup",
             "--output-root", str(output_root)],
            cwd=repo, check=True,
        )
    archive_base = drive_root / archive_name
    archive_path = archive_base.with_suffix(".zip")
    archive_path.unlink(missing_ok=True)
    output_archive = Path(shutil.make_archive(
        str(archive_base), "zip", root_dir=output_root, base_dir=owned_directory.name,
    ))
    print(f"Output archive: {output_archive}")
    print(f"Output SHA256: {hashlib.sha256(output_archive.read_bytes()).hexdigest()}")

if RUN_SCIENTIFIC_GRID:
    run_scientific("--run-scientific", "--validate-scientific-output",
                   output_root / "scientific_grid_dedup", "dipdeck_scientific_grid_dedup_outputs")
if RUN_SCIENTIFIC_REFERENCE:
    run_scientific("--run-scientific-reference", "--validate-scientific-reference-output",
                   output_root / "scientific_reference_dedup", "dipdeck_scientific_reference_dedup_outputs")
if not RUN_SCIENTIFIC_GRID and not RUN_SCIENTIFIC_REFERENCE:
    print("Scientific work disabled. Enable one scientific flag in the configuration cell.")
